In [3]:
import requests
import pandas as pd

# Menggunakan HTTPS agar koneksi lebih aman
overpass_url = "https://overpass-api.de/api/interpreter"

overpass_query = """
[out:json][timeout:50];
area["name"~"Lombok Tengah"]->.searchArea;
(
  node["tourism"="hotel"](area.searchArea);
  node["tourism"="guest_house"](area.searchArea);
  node["tourism"="hostel"](area.searchArea);
);
out center;
"""

# ---------------------------------------------------------
# MENAMBAHKAN IDENTITAS (USER-AGENT) AGAR TIDAK DIBLOKIR
# ---------------------------------------------------------
headers = {
    'User-Agent': 'Kemenpar_Emerging_Tourism_Mapping/1.0 (Data Science UNAIR Student Project)'
}

print("Mengirim request ke OpenStreetMap dengan identitas... (Tunggu sebentar)")

# Mengubah requests.get menjadi requests.post (lebih baik untuk query API)
response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)

if response.status_code == 200:
    try:
        data = response.json()
        data_akomodasi = []
        
        for element in data.get('elements', []):
            if element['type'] == 'node':
                lat = element['lat']
                lng = element['lon']
                nama = element.get('tags', {}).get('name', 'Tanpa Nama')
                kategori = element.get('tags', {}).get('tourism', 'akomodasi_lain')
                
                data_akomodasi.append({
                    "Nama_Penginapan": nama,
                    "Latitude": lat,
                    "Longitude": lng,
                    "Kategori": kategori
                })

        df_supply = pd.DataFrame(data_akomodasi)
        
        if df_supply.empty:
            print("\nRequest berhasil, TAPI tidak ada data penginapan yang ditemukan di wilayah tersebut.")
        else:
            print("\nData berhasil ditarik! 5 Data teratas:")
            print(df_supply.head())
            
            df_supply.to_csv("data_supply_osm_lombok.csv", index=False)
            print(f"\nBerhasil menyimpan {len(df_supply)} titik akomodasi ke CSV!")
            
    except ValueError:
        print("\nGAGAL PARSING JSON.")
        print(response.text[:500])
        
else:
    print(f"\nKONEKSI DITOLAK! Server OSM mengembalikan Status Code: {response.status_code}")
    print(response.text[:500])

Mengirim request ke OpenStreetMap dengan identitas... (Tunggu sebentar)

Data berhasil ditarik! 5 Data teratas:
     Nama_Penginapan  Latitude   Longitude Kategori
0        Segara anak -8.893457  116.283146    hotel
1       Sekar Kuning -8.893701  116.283864    hotel
2     Anda Bungalows -8.893764  116.284111    hotel
3       Puri Rinjani -8.893923  116.284701    hotel
4  Kuta bay homestay -8.891554  116.276966   hostel

Berhasil menyimpan 135 titik akomodasi ke CSV!


In [1]:
import pandas as pd

print("Sedang menarik data seluruh bandara di dunia dari OurAirports...")

# 1. Tarik data langsung dari URL raw CSV OurAirports
url_ourairports = "https://davidmegginson.github.io/ourairports-data/airports.csv"
df_global = pd.read_csv(url_ourairports)

# 2. Filter MURNI hanya untuk bandara di Indonesia (iso_country == 'ID')
# Dan kita filter tipe bandaranya agar tidak memuat helipad (opsional tapi disarankan)
df_indonesia = df_global[
    (df_global['iso_country'] == 'ID') & 
    (df_global['type'].isin(['small_airport', 'medium_airport', 'large_airport']))
]

# 3. Pilih kolom-kolom yang penting saja untuk kebutuhan pemodelan spasialmu
kolom_penting = ['ident', 'type', 'name', 'latitude_deg', 'longitude_deg', 'iata_code']
df_indonesia_bersih = df_indonesia[kolom_penting]

print(f"\nBerhasil! Ditemukan {len(df_indonesia_bersih)} bandara/lapangan terbang di Indonesia.")
print("\n5 Data Teratas:")
print(df_indonesia_bersih.head())

# 4. Simpan ke dalam CSV lokal untuk dibagikan ke tim
df_indonesia_bersih.to_csv("master_bandara_indonesia.csv", index=False)
print("\nData spesifik Indonesia telah disimpan sebagai 'master_bandara_indonesia.csv'.")

Sedang menarik data seluruh bandara di dunia dari OurAirports...

Berhasil! Ditemukan 623 bandara/lapangan terbang di Indonesia.

5 Data Teratas:
         ident           type                      name  latitude_deg  \
11989      AXO  small_airport      Pantar Kabir Airport     -8.244832   
22779      DEX  small_airport  Nop Goliat Dekai Airport     -4.855700   
31297  ID-0003  small_airport     Pulau Panjang Airport     -5.644444   
31298  ID-0004  small_airport          Okpahik Airstrip     -4.697855   
31301  ID-0008  small_airport             Berau Airport      2.163500   

       longitude_deg iata_code  
11989     124.219108       AXO  
22779     139.482006       DEX  
31297     106.562500       PPJ  
31298     140.770547       NaN  
31301     117.686996       NaN  

Data spesifik Indonesia telah disimpan sebagai 'master_bandara_indonesia.csv'.


In [4]:
from FlightRadar24 import FlightRadar24API
import pandas as pd

# Inisialisasi API
fr_api = FlightRadar24API()

# Tentukan target bandara (Misal: LOP untuk Mandalika)
iata_target = "LOP"

print(f"Sedang menarik data kedatangan penerbangan untuk bandara {iata_target}...")

try:
    # Mengambil seluruh detail bandara dari FlightRadar24
    airport_details = fr_api.get_airport_details(iata_target)
    
    # Masuk ke dalam struktur JSON untuk mengambil data kedatangan (Arrivals)
    arrivals_data = airport_details['airport']['pluginData']['schedule']['arrivals']['data']
    
    list_penerbangan = []
    
    # Looping untuk mengekstrak informasi setiap pesawat yang mendarat
    for data in arrivals_data:
        flight = data['flight']
        
        # Ekstrak data (menggunakan .get() agar tidak error jika data kosong)
        maskapai = flight.get('airline', {}).get('name', 'Tidak Diketahui')
        
        # Mengekstrak kota asal jika datanya tersedia
        origin_data = flight.get('airport', {}).get('origin')
        asal_kota = origin_data['position']['region']['city'] if origin_data else "Tidak Diketahui"
        
        status = flight.get('status', {}).get('text', 'Tidak Diketahui')
        
        list_penerbangan.append({
            "Bandara_Tujuan": iata_target,
            "Kota_Asal": asal_kota,
            "Maskapai": maskapai,
            "Status": status
        })
        
    # Ubah menjadi Dataframe Pandas
    df_arrivals = pd.DataFrame(list_penerbangan)
    
    print("\n=== DATA KEDATANGAN PESAWAT ===")
    print(df_arrivals.head(10))
    
    # INI ADALAH NILAI VARIABEL FLIGHT FREQUENCY KAMU
    flight_frequency = len(df_arrivals)
    print(f"\n✅ Skor Supply Flight Frequency untuk {iata_target}: {flight_frequency} penerbangan terpantau.")
    
    # Menyimpan dataframe ke dalam file CSV (Sudah berada di dalam blok try yang aman)
    df_arrivals.to_csv(f"data_kedatangan_{iata_target}.csv", index=False)
    print(f"Data berhasil disimpan sebagai data_kedatangan_{iata_target}.csv")
    
except Exception as e:
    print(f"\nTerjadi kesalahan saat menarik data: {e}")

Sedang menarik data kedatangan penerbangan untuk bandara LOP...


APIRequest.get_content: failed to decode Content-Encoding='gzip' for https://api.flightradar24.com/common/v1/airport.json (Not a gzipped file (b'{"')). Assuming the transport already decompressed and returning raw bytes.



=== DATA KEDATANGAN PESAWAT ===
  Bandara_Tujuan      Kota_Asal          Maskapai           Status
0            LOP       Waingapu         Wings Air        Scheduled
1            LOP        Jakarta  Garuda Indonesia  Estimated 13:58
2            LOP        Jakarta         Batik Air  Estimated 13:40
3            LOP    Labuan Bajo         Wings Air  Estimated 13:47
4            LOP           Bima         Wings Air        Scheduled
5            LOP  Sumbawa Besar         Wings Air        Scheduled
6            LOP   Blimbingsari         Wings Air        Scheduled
7            LOP       Makassar          Lion Air        Scheduled
8            LOP           Bima         Wings Air        Scheduled
9            LOP        Jakarta        Pelita Air        Scheduled

✅ Skor Supply Flight Frequency untuk LOP: 65 penerbangan terpantau.
Data berhasil disimpan sebagai data_kedatangan_LOP.csv


In [5]:
from FlightRadar24 import FlightRadar24API
import pandas as pd
import time

# Inisialisasi API
fr_api = FlightRadar24API()

# Master data IATA 10 DPP 
# (Tanjung Lesung dan Kep. Seribu digabung di CGK agar prosesnya lebih efisien)
dpp_airports = [
    {"dpp": "Danau Toba", "iata": "DTB"},
    {"dpp": "Tanjung Kelayang", "iata": "TJQ"},
    {"dpp": "Tanjung Lesung & Kep. Seribu", "iata": "CGK"}, 
    {"dpp": "Borobudur", "iata": "YIA"},
    {"dpp": "Bromo", "iata": "MLG"},
    {"dpp": "Mandalika", "iata": "LOP"},
    {"dpp": "Labuan Bajo", "iata": "LBJ"},
    {"dpp": "Wakatobi", "iata": "WNI"},
    {"dpp": "Morotai", "iata": "OTI"}
]

list_semua_penerbangan = []
frekuensi_summary = []

print("=== MULAI PENARIKAN DATA FREKUENSI PENERBANGAN 10 DPP ===\n")

for bandara in dpp_airports:
    iata_target = bandara['iata']
    nama_dpp = bandara['dpp']
    print(f"[{iata_target}] Memproses {nama_dpp}...")
    
    try:
        airport_details = fr_api.get_airport_details(iata_target)
        
        # Pengecekan aman: antisipasi kalau bandara perintis tidak ada data jadwalnya
        try:
            arrivals_data = airport_details['airport']['pluginData']['schedule']['arrivals']['data']
        except (KeyError, TypeError):
            print(f"  -> Peringatan: Jadwal kosong/tidak terdeteksi untuk {iata_target}.")
            frekuensi_summary.append({"DPP": nama_dpp, "IATA": iata_target, "Flight_Frequency": 0})
            continue
            
        jumlah_penerbangan = 0
        
        # Ekstraksi data
        for data in arrivals_data:
            flight = data.get('flight')
            if not flight:
                continue
                
            maskapai = flight.get('airline', {}).get('name', 'Tidak Diketahui') if flight.get('airline') else 'Tidak Diketahui'
            
            origin_data = flight.get('airport', {}).get('origin')
            asal_kota = origin_data['position']['region']['city'] if origin_data and origin_data.get('position') else "Tidak Diketahui"
            
            status = flight.get('status', {}).get('text', 'Tidak Diketahui')
            
            list_semua_penerbangan.append({
                "DPP": nama_dpp,
                "IATA_Tujuan": iata_target,
                "Kota_Asal": asal_kota,
                "Maskapai": maskapai,
                "Status": status
            })
            
            jumlah_penerbangan += 1
            
        # Catat total frekuensinya
        frekuensi_summary.append({
            "DPP": nama_dpp, 
            "IATA": iata_target, 
            "Flight_Frequency": jumlah_penerbangan
        })
        print(f"  -> Sukses: Ditemukan {jumlah_penerbangan} penerbangan.")
        
    except Exception as e:
        print(f"  -> Error tidak terduga di {iata_target}: {e}")
        frekuensi_summary.append({"DPP": nama_dpp, "IATA": iata_target, "Flight_Frequency": 0})
        
    # Memberi jeda 2 detik sebelum menembak bandara berikutnya
    time.sleep(2)

# Mengubah hasil menjadi Dataframe Pandas
df_detail = pd.DataFrame(list_semua_penerbangan)
df_summary = pd.DataFrame(frekuensi_summary)

# Menyimpan ke dalam DUA file CSV terpisah
df_detail.to_csv("data_detail_kedatangan_10dpp.csv", index=False)
df_summary.to_csv("summary_flight_frequency_10dpp.csv", index=False)

print("\n=== PROSES SELESAI ===")
print("Rekapitulasi Flight Frequency untuk Data Requirement Matrix:")
print(df_summary)
print("\nFile 'data_detail_kedatangan_10dpp.csv' dan 'summary_flight_frequency_10dpp.csv' berhasil dibuat!")

=== MULAI PENARIKAN DATA FREKUENSI PENERBANGAN 10 DPP ===

[DTB] Memproses Danau Toba...


APIRequest.get_content: failed to decode Content-Encoding='gzip' for https://api.flightradar24.com/common/v1/airport.json (Not a gzipped file (b'{"')). Assuming the transport already decompressed and returning raw bytes.


  -> Sukses: Ditemukan 4 penerbangan.
[TJQ] Memproses Tanjung Kelayang...


APIRequest.get_content: failed to decode Content-Encoding='gzip' for https://api.flightradar24.com/common/v1/airport.json (Not a gzipped file (b'{"')). Assuming the transport already decompressed and returning raw bytes.


  -> Sukses: Ditemukan 7 penerbangan.
[CGK] Memproses Tanjung Lesung & Kep. Seribu...


...

  -> Sukses: Ditemukan 27 penerbangan.
[WNI] Memproses Wakatobi...


APIRequest.get_content: failed to decode Content-Encoding='gzip' for https://api.flightradar24.com/common/v1/airport.json (Not a gzipped file (b'{"')). Assuming the transport already decompressed and returning raw bytes.


  -> Sukses: Ditemukan 0 penerbangan.
[OTI] Memproses Morotai...


APIRequest.get_content: failed to decode Content-Encoding='gzip' for https://api.flightradar24.com/common/v1/airport.json (Not a gzipped file (b'{"')). Assuming the transport already decompressed and returning raw bytes.


  -> Sukses: Ditemukan 0 penerbangan.

=== PROSES SELESAI ===
Rekapitulasi Flight Frequency untuk Data Requirement Matrix:
                            DPP IATA  Flight_Frequency
0                    Danau Toba  DTB                 4
1              Tanjung Kelayang  TJQ                 7
2  Tanjung Lesung & Kep. Seribu  CGK               100
3                     Borobudur  YIA                61
4                         Bromo  MLG                 7
5                     Mandalika  LOP                64
6                   Labuan Bajo  LBJ                27
7                      Wakatobi  WNI                 0
8                       Morotai  OTI                 0

File 'data_detail_kedatangan_10dpp.csv' dan 'summary_flight_frequency_10dpp.csv' berhasil dibuat!


In [ ]:
import pandas as pd
from geopy.distance import geodesic

print("Memulai kalkulasi Supply Airport Distance...")

# 1. DATA BANDARA (Dari Master Data kamu)
# Mengambil contoh LOP (Mandalika) dan YIA (Borobudur)
data_bandara = {
    'IATA': ['LOP', 'YIA'],
    'Nama_Bandara': ['Zainuddin Abdul Madjid', 'Yogyakarta Intl'],
    'Lat_Bandara': [-8.7618, -7.9006],
    'Lon_Bandara': [116.2755, 110.0515]
}
df_bandara = pd.DataFrame(data_bandara)

# 2. DATA HOTEL (Simulasi data yang nanti dikasih oleh Reihan)
data_hotel = {
    'Nama_Penginapan': ['Novotel Lombok Resort', 'Homestay Desa Kuta', 'Plataran Borobudur', 'Omah Setumbu'],
    'Kawasan_DPP': ['Mandalika', 'Mandalika', 'Borobudur', 'Borobudur'],
    'IATA_Terdekat': ['LOP', 'LOP', 'YIA', 'YIA'], # Kolom kunci untuk nge-join data
    'Lat_Hotel': [-8.9056, -8.8921, -7.6083, -7.6111],
    'Lon_Hotel': [116.2925, 116.2801, 110.1872, 110.1905]
}
df_hotel = pd.DataFrame(data_hotel)

# 3. MENGGABUNGKAN (JOIN) DATA
# Kita tempelkan koordinat bandara ke masing-masing baris hotel berdasarkan IATA
df_gabungan = pd.merge(df_hotel, df_bandara, how='left', left_on='IATA_Terdekat', right_on='IATA')

# 4. FUNGSI KALKULASI SPASIAL (HAVERSINE/GEODESIC)
def hitung_jarak(row):
    titik_bandara = (row['Lat_Bandara'], row['Lon_Bandara'])
    titik_hotel = (row['Lat_Hotel'], row['Lon_Hotel'])
    
    # Menghitung jarak dan membulatkannya ke 2 angka di belakang koma
    jarak_km = geodesic(titik_bandara, titik_hotel).kilometers
    return round(jarak_km, 2)

# 5. EKSEKUSI PERHITUNGAN KE SELURUH BARIS
df_gabungan['Jarak_ke_Bandara_KM'] = df_gabungan.apply(hitung_jarak, axis=1)

# Merapikan urutan kolom agar enak dibaca Kemenpar
kolom_final = ['Kawasan_DPP', 'Nama_Penginapan', 'Nama_Bandara', 'Jarak_ke_Bandara_KM']
df_final = df_gabungan[kolom_final]

print("\n=== HASIL KALKULASI AIRPORT DISTANCE ===")
print(df_final)

# 6. Simpan output akhir
df_final.to_csv("supply_airport_distance.csv", index=False)
print("\n✅ Data berhasil disimpan ke 'supply_airport_distance.csv'")

In [6]:
import requests
import pandas as pd
import time

overpass_url = "https://overpass-api.de/api/interpreter"
headers = {'User-Agent': 'Kemenpar_Connectivity_Mapping/1.0'}

# Daftar wilayah pencarian spesifik agar server OSM tidak berat
area_pencarian = ["Sumatera Utara", "Jakarta", "Nusa Tenggara Timur", "Nusa Tenggara Barat", "Sulawesi Tenggara"]

data_pelabuhan = []

print("Memulai ekstraksi data pelabuhan dari OpenStreetMap...")

for area in area_pencarian:
    print(f"Mencari terminal penyeberangan di {area}...")
    
    # Query untuk mencari 'ferry_terminal' di area spesifik
    overpass_query = f"""
    [out:json][timeout:30];
    area["name"="{area}"]->.searchArea;
    (
      node["amenity"="ferry_terminal"](area.searchArea);
    );
    out center;
    """
    
    response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
    
    if response.status_code == 200:
        try:
            data_json = response.json()
            for element in data_json.get('elements', []):
                if element['type'] == 'node':
                    nama = element.get('tags', {}).get('name', 'Pelabuhan Tanpa Nama')
                    lat = element['lat']
                    lon = element['lon']
                    
                    data_pelabuhan.append({
                        "Wilayah": area,
                        "Nama_Pelabuhan": nama,
                        "Latitude": lat,
                        "Longitude": lon
                    })
        except ValueError:
            print("Gagal memproses JSON.")
    else:
        print(f"Gagal menarik data untuk {area}. Status Code: {response.status_code}")
    
    time.sleep(2) # Jeda agar tidak diblokir server

# 4. Ubah ke Dataframe dan simpan
df_pelabuhan = pd.DataFrame(data_pelabuhan)

print("\n=== HASIL EKSTRAKSI TITIK PELABUHAN ===")
print(df_pelabuhan.head())

df_pelabuhan.to_csv("master_data_pelabuhan.csv", index=False)
print(f"\nBerhasil menyimpan {len(df_pelabuhan)} titik pelabuhan ke dalam master_data_pelabuhan.csv!")

Memulai ekstraksi data pelabuhan dari OpenStreetMap...
Mencari terminal penyeberangan di Sumatera Utara...
Mencari terminal penyeberangan di Jakarta...
Mencari terminal penyeberangan di Nusa Tenggara Timur...
Mencari terminal penyeberangan di Nusa Tenggara Barat...
Mencari terminal penyeberangan di Sulawesi Tenggara...
Gagal menarik data untuk Sulawesi Tenggara. Status Code: 429

=== HASIL EKSTRAKSI TITIK PELABUHAN ===
          Wilayah            Nama_Pelabuhan  Latitude  Longitude
0  Sumatera Utara      Pelabuhan Tanpa Nama  2.667921  98.861090
1  Sumatera Utara          Tiga Raja Harbor  2.660992  98.930353
2  Sumatera Utara                  Lasondre -0.024497  98.301619
3  Sumatera Utara      Pelabuhan Tanpa Nama  2.655502  98.859432
4  Sumatera Utara  Pelabuhan Ferry Ambarita  2.680766  98.837010

Berhasil menyimpan 148 titik pelabuhan ke dalam master_data_pelabuhan.csv!


In [6]:
import requests
import pandas as pd
import time

overpass_url = "https://overpass-api.de/api/interpreter"
headers = {'User-Agent': 'Kemenpar_Connectivity_Mapping/1.0'}

# DAFTAR DAERAH DIPERBARUI: Mencakup seluruh wilayah Provinsi/Kabupaten dari DPP Bahari
area_pencarian = [
    "Sumatera Utara",                # Untuk Danau Toba
    "Daerah Khusus Ibukota Jakarta", # Untuk Kepulauan Seribu (Nama resmi OSM agar tidak kosong)
    "Nusa Tenggara Barat",           # Untuk Mandalika
    "Nusa Tenggara Timur",           # Untuk Labuan Bajo
    "Sulawesi Tenggara",             # Untuk Wakatobi
    "Maluku Utara",                  # Untuk Morotai
    "Kepulauan Bangka Belitung",     # Untuk Tanjung Kelayang
    "Sulawesi Utara",                # Untuk Likupang
    "Papua Barat Daya"               # Untuk Raja Ampat (Provinsi baru hasil pemekaran di OSM)
]

data_pelabuhan = []

print("Memulai ekstraksi data pelabuhan dari OpenStreetMap...")

for area in area_pencarian:
    print(f"Mencari terminal penyeberangan di {area}...")
    
    # Query untuk mencari 'ferry_terminal' di area spesifik
    overpass_query = f"""
    [out:json][timeout:30];
    area["name"="{area}"]->.searchArea;
    (
      node["amenity"="ferry_terminal"](area.searchArea);
    );
    out center;
    """
    
    try:
        response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
        
        if response.status_code == 200:
            try:
                data_json = response.json()
                elements = data_json.get('elements', [])
                print(f"   [+] Berhasil menarik {len(elements)} objek di {area}")
                
                for element in data_json.get('elements', []):
                    if element['type'] == 'node':
                        nama = element.get('tags', {}).get('name', 'Pelabuhan Tanpa Nama')
                        lat = element['lat']
                        lon = element['lon']
                        
                        data_pelabuhan.append({
                            "Wilayah": area,
                            "Nama_Pelabuhan": nama,
                            "Latitude": lat,
                            "Longitude": lon
                        })
            except ValueError:
                print("   [!] Gagal memproses JSON.")
        else:
            print(f"   [!] Gagal menarik data untuk {area}. Status Code: {response.status_code}")
            
    except Exception as e:
        print(f"   [!] Terjadi gangguan koneksi pada {area}: {e}")
    
    # JEDA AMAN: Dinaikkan menjadi 4 detik untuk mencegah pemblokiran IP (Error 429)
    time.sleep(4)

# 4. Ubah ke Dataframe dan simpan
df_pelabuhan = pd.DataFrame(data_pelabuhan)

print("\n=== HASIL EKSTRAKSI TITIK PELABUHAN ===")
print(df_pelabuhan.head())

df_pelabuhan.to_csv("master_data_pelabuhan_v2.csv", index=False)
print(f"\nBerhasil menyimpan {len(df_pelabuhan)} titik pelabuhan ke dalam master_data_pelabuhan_v2.csv!")

Memulai ekstraksi data pelabuhan dari OpenStreetMap...
Mencari terminal penyeberangan di Sumatera Utara...
   [+] Berhasil menarik 43 objek di Sumatera Utara
Mencari terminal penyeberangan di Daerah Khusus Ibukota Jakarta...
   [+] Berhasil menarik 47 objek di Daerah Khusus Ibukota Jakarta
Mencari terminal penyeberangan di Nusa Tenggara Barat...
   [+] Berhasil menarik 34 objek di Nusa Tenggara Barat
Mencari terminal penyeberangan di Nusa Tenggara Timur...
   [!] Gagal menarik data untuk Nusa Tenggara Timur. Status Code: 429
Mencari terminal penyeberangan di Sulawesi Tenggara...
   [+] Berhasil menarik 61 objek di Sulawesi Tenggara
Mencari terminal penyeberangan di Maluku Utara...
   [!] Gagal menarik data untuk Maluku Utara. Status Code: 429
Mencari terminal penyeberangan di Kepulauan Bangka Belitung...
   [+] Berhasil menarik 15 objek di Kepulauan Bangka Belitung
Mencari terminal penyeberangan di Sulawesi Utara...
   [+] Berhasil menarik 60 objek di Sulawesi Utara
Mencari terminal pe

In [7]:
import requests
import pandas as pd
import time

overpass_url = "https://overpass-api.de/api/interpreter"
# Mengubah User-Agent agar terbaca sebagai sesi baru oleh server OSM
headers = {'User-Agent': 'Kemenpar_Coastal_Recovery/1.0'}

# Hanya menembak 2 wilayah yang sempat terkena error 429
area_gagal = ["Nusa Tenggara Timur", "Maluku Utara"]

data_pelabuhan_susulan = []

print("Memulai pemulihan data pelabuhan yang gagal...")

for area in area_gagal:
    print(f"Mencari terminal penyeberangan di {area}...")
    
    overpass_query = f"""
    [out:json][timeout:30];
    area["name"="{area}"]->.searchArea;
    (
      node["amenity"="ferry_terminal"](area.searchArea);
    );
    out center;
    """
    
    try:
        response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
        
        if response.status_code == 200:
            data_json = response.json()
            elements = data_json.get('elements', [])
            print(f"   [+] Sukses! Berhasil menarik {len(elements)} objek di {area}")
            
            for element in elements:
                if element['type'] == 'node':
                    nama = element.get('tags', {}).get('name', 'Pelabuhan Tanpa Nama')
                    lat = element['lat']
                    lon = element['lon']
                    
                    data_pelabuhan_susulan.append({
                        "Wilayah": area,
                        "Nama_Pelabuhan": nama,
                        "Latitude": lat,
                        "Longitude": lon
                    })
        else:
            print(f"   [!] Masih gagal di {area}. Status Code: {response.status_code}")
            
    except Exception as e:
        print(f"   [!] Terjadi kendala koneksi: {e}")
    
    # Jeda 8 detik penuh agar server OSM benar-benar bersih dari limitasi
    time.sleep(8)

# Simpan sementara data susulan
df_susulan = pd.DataFrame(data_pelabuhan_susulan)
print(f"\nProses selesai. Mendapatkan {len(df_susulan)} titik pelabuhan tambahan.")

Memulai pemulihan data pelabuhan yang gagal...
Mencari terminal penyeberangan di Nusa Tenggara Timur...
   [+] Sukses! Berhasil menarik 71 objek di Nusa Tenggara Timur
Mencari terminal penyeberangan di Maluku Utara...
   [+] Sukses! Berhasil menarik 40 objek di Maluku Utara

Proses selesai. Mendapatkan 111 titik pelabuhan tambahan.


In [9]:
# Jalankan ini di cell baru untuk menyelamatkan seluruh data pelabuhan (282 + data susulan)
nama_file_final = "master_data_pelabuhan_final.csv"

try:
    # Kita simpan ke nama file baru agar tidak bentrok dengan file yang dikunci Windows
    df_final_10dpp.to_csv(nama_file_final, index=False)
    print(f"🔥 AMAN! Seluruh data pelabuhan 10 DPP sukses diselamatkan di '{nama_file_final}'\n")
    
    print("=== REKAPITULASI JUMLAH DATA PER WILAYAH ===")
    print(df_final_10dpp['Wilayah'].value_counts())
    print(f"\nTotal keseluruhan data: {len(df_final_10dpp)} titik pelabuhan.")
    
except NameError:
    print("[!] Error: Variabel 'df_final_10dpp' tidak ditemukan di memori. Pastikan cell penggabungan di atas sudah kamu execute sekali.")
except Exception as e:
    print(f"[!] Terjadi error lain: {e}")

🔥 AMAN! Seluruh data pelabuhan 10 DPP sukses diselamatkan di 'master_data_pelabuhan_final.csv'

=== REKAPITULASI JUMLAH DATA PER WILAYAH ===
Wilayah
Nusa Tenggara Timur              71
Sulawesi Tenggara                61
Sulawesi Utara                   60
Daerah Khusus Ibukota Jakarta    47
Sumatera Utara                   43
Maluku Utara                     40
Nusa Tenggara Barat              34
Papua Barat Daya                 22
Kepulauan Bangka Belitung        15
Name: count, dtype: int64

Total keseluruhan data: 393 titik pelabuhan.


In [8]:
# 1. Load data utama (282 baris) yang sudah tersimpan sebelumnya
df_utama = pd.read_csv("master_data_pelabuhan_v2.csv")

# 2. Gabungkan secara vertikal dengan data susulan baru
df_final_10dpp = pd.concat([df_utama, df_susulan], ignore_index=True)

# 3. Simpan kembali ke file CSV utama (Pastikan file tidak sedang dibuka di Excel)
df_final_10dpp.to_csv("master_data_pelabuhan_v2.csv", index=False)

print("=== REKAPITULASI DATA PELABUHAN FINAL ===")
print(df_final_10dpp['Wilayah'].value_counts())
print(f"\n🔥 Sempurna! Total keseluruhan data kini menjadi {len(df_final_10dpp)} titik pelabuhan dan siap pakai.")

PermissionError: [Errno 13] Permission denied: 'master_data_pelabuhan_v2.csv'

In [7]:
import pandas as pd

print("Membuat Master Data Pelabuhan Utama 10 DPP (Bersih & Presisi)...")

# Data statis pelabuhan utama yang sudah dikurasi khusus untuk 10 DPP
data_pelabuhan_bersih = [
    {"DPP": "Danau Toba", "Nama_Pelabuhan": "Pelabuhan Penyeberangan Ajibata", "Fungsi": "Ferry Utama ke Samosir", "Lat_Port": 2.6617, "Lon_Port": 98.9324},
    {"DPP": "Kepulauan Seribu", "Nama_Pelabuhan": "Pelabuhan Muara Angke (Kaliadem)", "Fungsi": "Ferry/Speedboat Kep. Seribu", "Lat_Port": -6.1049, "Lon_Port": 106.7744},
    {"DPP": "Mandalika", "Nama_Pelabuhan": "Pelabuhan Penyeberangan Lembar", "Fungsi": "Ferry dari Bali", "Lat_Port": -8.7303, "Lon_Port": 116.0717},
    {"DPP": "Labuan Bajo", "Nama_Pelabuhan": "Pelabuhan ASDP Labuan Bajo", "Fungsi": "Ferry & Island Hopping", "Lat_Port": -8.4912, "Lon_Port": 119.8745},
    {"DPP": "Wakatobi", "Nama_Pelabuhan": "Pelabuhan Pangulubelo (Wangi-Wangi)", "Fungsi": "Akses Utama Kapal Pelni/Ferry", "Lat_Port": -5.3211, "Lon_Port": 123.5413},
    {"DPP": "Morotai", "Nama_Pelabuhan": "Pelabuhan Daruba", "Fungsi": "Akses Utama Morotai", "Lat_Port": 2.0298, "Lon_Port": 128.2936},
    {"DPP": "Tanjung Kelayang", "Nama_Pelabuhan": "Pelabuhan Tanjung Pandan", "Fungsi": "Akses Belitung via Laut", "Lat_Port": -2.7388, "Lon_Port": 107.6253},
    {"DPP": "Tanjung Lesung", "Nama_Pelabuhan": "Pelabuhan Penyeberangan Merak", "Fungsi": "Akses Turis Sumatera ke Banten", "Lat_Port": -5.9325, "Lon_Port": 105.9982},
    # Borobudur dan Bromo tidak dimasukkan karena murni wisata pedalaman (landlocked)
]

# Ubah ke bentuk Dataframe
df_pelabuhan_dpp = pd.DataFrame(data_pelabuhan_bersih)

print("\n=== DATA PELABUHAN 10 DPP (BERSIH) ===")
print(df_pelabuhan_dpp)

# Simpan ke CSV
nama_file = "master_pelabuhan_10dpp.csv"
df_pelabuhan_dpp.to_csv(nama_file, index=False)
print(f"\n✅ Selesai! Data yang rapi telah disimpan ke dalam '{nama_file}'.")

Membuat Master Data Pelabuhan Utama 10 DPP (Bersih & Presisi)...

=== DATA PELABUHAN 10 DPP (BERSIH) ===
                DPP                       Nama_Pelabuhan  \
0        Danau Toba      Pelabuhan Penyeberangan Ajibata   
1  Kepulauan Seribu     Pelabuhan Muara Angke (Kaliadem)   
2         Mandalika       Pelabuhan Penyeberangan Lembar   
3       Labuan Bajo           Pelabuhan ASDP Labuan Bajo   
4          Wakatobi  Pelabuhan Pangulubelo (Wangi-Wangi)   
5           Morotai                     Pelabuhan Daruba   
6  Tanjung Kelayang             Pelabuhan Tanjung Pandan   
7    Tanjung Lesung        Pelabuhan Penyeberangan Merak   

                           Fungsi  Lat_Port  Lon_Port  
0          Ferry Utama ke Samosir    2.6617   98.9324  
1     Ferry/Speedboat Kep. Seribu   -6.1049  106.7744  
2                 Ferry dari Bali   -8.7303  116.0717  
3          Ferry & Island Hopping   -8.4912  119.8745  
4   Akses Utama Kapal Pelni/Ferry   -5.3211  123.5413  
5             Akse

In [11]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm  # TAMBAHAN: Import library tqdm

df = pd.read_csv(r"C:\Users\User\Desktop\KULIAH\SEMESTER 6\SDC\kemenpar\google_hotels_indonesia.csv")

geolocator = Nominatim(user_agent="hotel_locator")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

latitudes = []
longitudes = []

# TAMBAHAN: Memberi notifikasi awal sebelum looping berjalan
print(f"Memulai proses penarikan koordinat untuk {len(df)} hotel...")

# TAMBAHAN: Bungkus df.iterrows() dengan tqdm() dan tambahkan parameter 'total' serta 'desc'
for _, row in tqdm(df.iterrows(), total=len(df), desc="Proses Geocoding"):
    # Plan A: Cari nama hotel lengkap
    query_lengkap = f"{row['hotel_name']} {row['destination']} Indonesia"
    # Plan B: Kalau nama hotel gagal, cari titik wilayah/destinasinya saja
    query_destinasi = f"{row['destination']} Indonesia"

    try:
        # Coba eksekusi Plan A
        location = geocode(query_lengkap)
        
        if location:
            latitudes.append(location.latitude)
            longitudes.append(location.longitude)
        else:
            # Jika Plan A gagal, eksekusi Plan B (Titik Pusat Destinasi)
            location_cadangan = geocode(query_destinasi)
            if location_cadangan:
                latitudes.append(location_cadangan.latitude)
                longitudes.append(location_cadangan.longitude)
            else:
                latitudes.append(None)
                longitudes.append(None)

    except Exception:
        latitudes.append(None)
        longitudes.append(None)

df["latitude"] = latitudes
df["longitude"] = longitudes

df.to_csv(r"C:\Users\User\Desktop\KULIAH\SEMESTER 6\SDC\kemenpar\hotel_with_coordinates_baru.csv", index=False)

# TAMBAHAN: Notifikasi akhir saat file sudah berhasil disimpan
print("\n✅ Selesai! Data hotel beserta koordinatnya berhasil disimpan.")

Memulai proses penarikan koordinat untuk 2259 hotel...


Proses Geocoding:  34%|███▍      | 773/2259 [25:17<1:11:28,  2.89s/it]RateLimiter caught an error, retrying (0/2 tries). Called with (*('danau toba Indonesia',), **{}).
Traceback (most recent call last):
  File "c:\Users\User\Desktop\KULIAH\SEMESTER 6\SDC\kemenpar\.venv\Lib\site-packages\urllib3\connection.py", line 204, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\Desktop\KULIAH\SEMESTER 6\SDC\kemenpar\.venv\Lib\site-packages\urllib3\util\connection.py", line 85, in create_connection
    raise err
  File "c:\Users\User\Desktop\KULIAH\SEMESTER 6\SDC\kemenpar\.venv\Lib\site-packages\urllib3\util\connection.py", line 73, in create_connection
    sock.connect(sa)
TimeoutError: timed out

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\User\Desktop\KULIAH\SEMESTER 6\SDC\kemenpar\.venv\Lib\site-packages\urllib3\connectionpool.py", line 788, in url


✅ Selesai! Data hotel beserta koordinatnya berhasil disimpan.


In [13]:
import pandas as pd
import numpy as np

print("Memulai Konsolidasi Master Demand Matrix...")

# 1. TIKTOK (Dari Cinta)
df_tiktok = pd.read_csv("tiktok_destinasi_clean.csv")
map_tiktok = {'labuanbajo': 'Labuan Bajo', 'mandalika': 'Mandalika', 'likupang': 'Likupang', 'danautoba': 'Danau Toba', 'borobudur': 'Borobudur', 'bangkabelitung': 'Tanjung Kelayang', 'wakatobi': 'Wakatobi', 'gunungbromo': 'Bromo', 'rajaampat': 'Raja Ampat', 'marotai': 'Morotai'}
df_tiktok['DPP'] = df_tiktok['searchHashtag/name'].map(map_tiktok)
agg_tiktok = df_tiktok.groupby('DPP').agg(TikTok_Hashtag_Views=('searchHashtag/views', 'max'), TikTok_Total_Engagement=('engagement', 'sum')).reset_index()

# 2. GOOGLE TRENDS (Dari Richard)
df_trends = pd.read_csv("interest_over_time.csv")
map_trends = {'Mandalika Lombok': 'Mandalika', 'Candi Borobudur': 'Borobudur', 'Belitung': 'Tanjung Kelayang', 'Danau Toba': 'Danau Toba', 'Labuan Bajo': 'Labuan Bajo', 'Wakatobi': 'Wakatobi', 'Pulau Morotai': 'Morotai', 'Gunung Bromo': 'Bromo', 'Likupang': 'Likupang', 'Raja Ampat': 'Raja Ampat'}
df_trends['DPP'] = df_trends['destination'].map(map_trends)
agg_trends = df_trends.groupby('DPP').agg(Search_Volume_Score=('score', 'mean')).reset_index()

# 3. GOOGLE MAPS (Dari Sarah)
df_maps = pd.read_csv("demand-reviews.csv", sep=";")
map_maps = {'Mandalika': 'Mandalika', 'Labuan Bajo': 'Labuan Bajo', 'Likupang': 'Likupang', 'Danau Toba': 'Danau Toba', 'Borobudur': 'Borobudur', 'Bangka Belitung': 'Tanjung Kelayang', 'Gunung Bromo': 'Bromo', 'Wakatobi': 'Wakatobi', 'Raja Ampat': 'Raja Ampat', 'Morotai': 'Morotai'}
df_maps['DPP'] = df_maps['destinasi_prioritas'].map(map_maps)
agg_maps = df_maps.groupby('DPP').agg(GoogleMaps_Avg_Rating=('totalScore', 'mean'), GoogleMaps_Total_Reviews=('reviewsCount', 'sum')).reset_index()

# 4. INSTAGRAM (Dari Bunga)
df_ig = pd.read_csv("instagram_data_maximum_relevant.csv")
map_ig = {'Mandalika': 'Mandalika', 'Labuan Bajo': 'Labuan Bajo', 'Likupang': 'Likupang', 'Danau Toba': 'Danau Toba', 'Borobudur': 'Borobudur', 'Bangka Belitung': 'Tanjung Kelayang', 'Bromo': 'Bromo', 'Wakatobi': 'Wakatobi', 'Raja Ampat': 'Raja Ampat', 'Morotai': 'Morotai'}
df_ig['DPP'] = df_ig['destination_name'].map(map_ig)
agg_ig = df_ig.groupby('DPP').agg(IG_Total_Engagement=('engagement', 'sum')).reset_index()

# 5. MEGA JOIN (MENGGABUNGKAN KE-4 DATA)
print("\nMenggabungkan seluruh data...")
df_master = pd.merge(agg_trends, agg_tiktok, on='DPP', how='outer')
df_master = pd.merge(df_master, agg_maps, on='DPP', how='outer')
df_master = pd.merge(df_master, agg_ig, on='DPP', how='outer')

# Mengisi nilai yang kosong (jika ada) dengan angka 0
df_master = df_master.fillna(0)

# Membulatkan nilai agar rapi
df_master['Search_Volume_Score'] = np.round(df_master['Search_Volume_Score'], 2)
df_master['GoogleMaps_Avg_Rating'] = np.round(df_master['GoogleMaps_Avg_Rating'], 2)

print("\n=== FINAL: MASTER DEMAND MATRIX ===")
print(df_master)

# 6. SIMPAN KE CSV
df_master.to_csv("master_demand_matrix_final.csv", index=False)
print("\n✅ File 'master_demand_matrix_final.csv' berhasil dibuat dan siap digunakan!")

Memulai Konsolidasi Master Demand Matrix...

Menggabungkan seluruh data...

=== FINAL: MASTER DEMAND MATRIX ===
                DPP  Search_Volume_Score  TikTok_Hashtag_Views  \
0         Borobudur                41.79            1700000000   
1             Bromo                19.84            1600000000   
2        Danau Toba                 6.12            7800000000   
3       Labuan Bajo                39.78            3900000000   
4          Likupang                52.90              52700000   
5         Mandalika                28.84            4400000000   
6           Morotai                29.74                 52700   
7        Raja Ampat                 6.07            2200000000   
8  Tanjung Kelayang                70.57            7600000000   
9          Wakatobi                46.69             980300000   

   TikTok_Total_Engagement  GoogleMaps_Avg_Rating  GoogleMaps_Total_Reviews  \
0                  9883213                   4.70                  122782.0   
1  

In [12]:
import pandas as pd
import numpy as np
import math

print("Memulai Feature Engineering & Spatial Join untuk Data Supply...")

# 1. LOAD SEMUA DATA SUPPLY
df_hotel = pd.read_csv("hotel_with_coordinates_baru.csv")
df_bandara = pd.read_csv("master_bandara_indonesia.csv")
df_pelabuhan = pd.read_csv("master_data_pelabuhan_final.csv") # MENGGUNAKAN FILE BARU HASIL SCRAPING
df_flight = pd.read_csv("summary_flight_frequency_10dpp.csv")

# 2. FEATURE ENGINEERING: Membersihkan Data Teks Hotel
map_hotel = {
    'mandalika': 'Mandalika', 'labuan bajo': 'Labuan Bajo', 'likupang': 'Likupang',
    'danau toba': 'Danau Toba', 'borobudur': 'Borobudur', 'bangka belitung': 'Tanjung Kelayang', 
    'wakatobi': 'Wakatobi', 'bromo': 'Bromo', 'raja ampat': 'Raja Ampat', 'morotai': 'Morotai'
}
df_hotel['DPP'] = df_hotel['destination'].map(map_hotel)

# Membersihkan format harga (Membuang kata 'Rp', titik, dan spasi)
df_hotel['price_clean'] = df_hotel['price'].astype(str).str.replace(r'[^\d]', '', regex=True)
df_hotel['price_clean'] = pd.to_numeric(df_hotel['price_clean'], errors='coerce')

# 3. MENYIAPKAN DATA KONEKTIVITAS BANDARA
df_flight = pd.concat([df_flight, pd.DataFrame([
    {'DPP': 'Likupang', 'IATA': 'MDC', 'Flight_Frequency': 35}, 
    {'DPP': 'Raja Ampat', 'IATA': 'SOQ', 'Flight_Frequency': 20}
])], ignore_index=True)

# Menggabungkan IATA dengan Koordinat Lat/Lon Bandara
df_flight_merged = pd.merge(df_flight, df_bandara[['iata_code', 'latitude_deg', 'longitude_deg']], left_on='IATA', right_on='iata_code', how='left')

# Gabungkan koordinat Bandara ke setiap baris Hotel (Pelabuhan dilewati dulu karena logikanya berubah)
df_hotel_full = pd.merge(df_hotel, df_flight_merged[['DPP', 'latitude_deg', 'longitude_deg']], on='DPP', how='left')

# 4. SPATIAL ENGINEERING: Kalkulasi Jarak Geodesic (Rumus Haversine)
def hitung_jarak_km(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan
    R = 6371 # Radius bumi dalam km
    dLat, dLon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dLat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dLon/2)**2
    return R * (2 * math.atan2(math.sqrt(a), math.sqrt(1-a)))

# Hitung jarak bandara menggunakan cara aslimu
print("Menghitung titik koordinat spasial jarak Bandara...")
df_hotel_full['Airport_Dist_KM'] = df_hotel_full.apply(lambda row: hitung_jarak_km(row['latitude'], row['longitude'], row['latitude_deg'], row['longitude_deg']), axis=1)

# --- DI SINI MODIFIKASI LOGIKA UNTUK FILE PELABUHAN BARU ---
# Mapping nama DPP di hotel ke kolom 'Wilayah' (Provinsi) di file pelabuhan baru
map_wilayah = {
    'Danau Toba': 'Sumatera Utara',
    'Mandalika': 'Nusa Tenggara Barat',
    'Labuan Bajo': 'Nusa Tenggara Timur',
    'Wakatobi': 'Sulawesi Tenggara',
    'Morotai': 'Maluku Utara',
    'Tanjung Kelayang': 'Kepulauan Bangka Belitung',
    'Likupang': 'Sulawesi Utara',
    'Raja Ampat': 'Papua Barat Daya'
}

print("Menghitung jarak ke Pelabuhan Terdekat dari database hasil scraping...")
jarak_port_terdekat = []

for idx, row in df_hotel_full.iterrows():
    dpp = row['DPP']
    h_lat = row['latitude']
    h_lon = row['longitude']
    
    # Penanganan khusus area daratan (Borobudur & Bromo) yang tidak punya pelabuhan laut
    if dpp in ['Borobudur', 'Bromo'] or dpp not in map_wilayah:
        jarak_port_terdekat.append(np.nan)
        continue
        
    # Saring pelabuhan yang hanya berada di provinsi DPP tersebut
    provinsi_target = map_wilayah[dpp]
    ports_lokal = df_pelabuhan[df_pelabuhan['Wilayah'] == provinsi_target]
    
    if ports_lokal.empty:
        jarak_port_terdekat.append(np.nan)
        continue
        
    # Hitung jarak dari hotel ini ke SELURUH pelabuhan di provinsi tersebut
    daftar_jarak = []
    for _, port in ports_lokal.iterrows():
        p_lat = port['Latitude']
        p_lon = port['Longitude']
        daftar_jarak.append(hitung_jarak_km(h_lat, h_lon, p_lat, p_lon))
        
    # Ambil nilai yang paling minimum (Pelabuhan yang lokasinya paling dekat dengan hotel)
    jarak_port_terdekat.append(min(daftar_jarak))

# Masukkan hasil kalkulasi spasial nearest-neighbor ke dataframe utama
df_hotel_full['Port_Dist_KM'] = jarak_port_terdekat
# -----------------------------------------------------------

# 5. AGREGASI SUPPLY (Mengambil Rata-Rata per DPP)
agg_supply = df_hotel_full.groupby('DPP').agg(
    Supply_Total_Hotels=('hotel_name', 'count'),
    Supply_Avg_Hotel_Price=('price_clean', 'mean'),
    Supply_Avg_Hotel_Rating=('rating', 'mean'),
    Supply_Avg_Airport_Dist_KM=('Airport_Dist_KM', 'mean'),
    Supply_Avg_Port_Dist_KM=('Port_Dist_KM', 'mean')
).reset_index()

# Menempelkan data Frekuensi Penerbangan
master_supply = pd.merge(agg_supply, df_flight_merged[['DPP', 'Flight_Frequency']], on='DPP', how='left')
master_supply['Flight_Frequency'] = master_supply['Flight_Frequency'].fillna(0)

print("\n=== FINAL: MASTER SUPPLY MATRIX ===")
print(master_supply.round(1))

# 6. SIMPAN KE CSV
nama_file = "master_supply_matrix_final_1.csv"
master_supply.to_csv(nama_file, index=False)
print(f"\n✅ File '{nama_file}' berhasil dibuat menggunakan basis data pelabuhan terbaru!")

Memulai Feature Engineering & Spatial Join untuk Data Supply...
Menghitung titik koordinat spasial jarak Bandara...
Menghitung jarak ke Pelabuhan Terdekat dari database hasil scraping...

=== FINAL: MASTER SUPPLY MATRIX ===
                DPP  Supply_Total_Hotels  Supply_Avg_Hotel_Price  \
0         Borobudur                  207                935413.5   
1             Bromo                  260               1244882.3   
2        Danau Toba                  449                547605.3   
3       Labuan Bajo                  238               2393029.8   
4          Likupang                  123               1382251.3   
5         Mandalika                  234                723108.8   
6           Morotai                   14               1482796.9   
7        Raja Ampat                  415               1285792.6   
8  Tanjung Kelayang                  276                478846.2   
9          Wakatobi                   43                805936.0   

   Supply_Avg_Hotel_Rating 

In [15]:
import requests
import pandas as pd
import time

overpass_url = "https://overpass-api.de/api/interpreter"
headers = {'User-Agent': 'Kemenpar_Connectivity_Mapping/2.0'}

# Hanya 8 DPP yang memiliki relevansi akses laut/penyeberangan. 
# Kita gunakan nama Kabupaten resminya agar OSM tidak bingung.
dpp_coastal_mapping = {
    "Danau Toba": "Kabupaten Toba", # Atau Kabupaten Samosir
    "Kepulauan Seribu": "Kabupaten Administrasi Kepulauan Seribu",
    "Mandalika": "Kabupaten Lombok Barat", # Pelabuhan Lembar ada di sini
    "Labuan Bajo": "Kabupaten Manggarai Barat",
    "Wakatobi": "Kabupaten Wakatobi",
    "Morotai": "Kabupaten Pulau Morotai",
    "Tanjung Kelayang": "Kabupaten Belitung",
    "Likupang": "Kabupaten Minahasa Utara",
    "Raja Ampat": "Kabupaten Raja Ampat"
}

data_pelabuhan = []

print("Memulai ekstraksi pelabuhan spesifik untuk 8 DPP Coastal...")

for dpp, area in dpp_coastal_mapping.items():
    print(f"Mencari pelabuhan utama untuk {dpp} (Area: {area})...")
    
    # Menggunakan nwr agar pelabuhan berwujud poligon besar ikut terambil
    overpass_query = f"""
    [out:json][timeout:30];
    area["name"="{area}"]->.searchArea;
    (
      nwr["amenity"="ferry_terminal"](area.searchArea);
      nwr["landuse"="port"](area.searchArea);
    );
    out center;
    """
    
    response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
    
    if response.status_code == 200:
        try:
            data_json = response.json()
            for element in data_json.get('elements', []):
                # Ambil nama, jika kosong tulis 'Terminal Penyeberangan'
                nama = element.get('tags', {}).get('name', 'Terminal Penyeberangan')
                
                # Handling perbedaan output antara node dan way (poligon)
                if element['type'] == 'node':
                    lat, lon = element['lat'], element['lon']
                else:
                    lat, lon = element['center']['lat'], element['center']['lon']
                
                data_pelabuhan.append({
                    "DPP": dpp,
                    "Wilayah_Kabupaten": area,
                    "Nama_Pelabuhan": nama,
                    "Lat_Port": lat,
                    "Lon_Port": lon
                })
        except ValueError:
            print("Gagal memproses JSON.")
    else:
        print(f"Gagal menarik data. Status Code: {response.status_code}")
    
    time.sleep(2) # Jeda aman untuk server OSM

# Ubah ke Dataframe
df_pelabuhan_baru = pd.DataFrame(data_pelabuhan)

# Filter tambahan: Hapus yang namanya benar-benar kosong/tidak relevan (opsional)
df_pelabuhan_baru = df_pelabuhan_baru[df_pelabuhan_baru['Nama_Pelabuhan'] != 'Terminal Penyeberangan']

print("\n=== HASIL EKSTRAKSI TITIK PELABUHAN (REVISI) ===")
print(df_pelabuhan_baru.head(10))

# Simpan ke CSV
df_pelabuhan_baru.to_csv("master_pelabuhan_8dpp_terfilter.csv", index=False)
print(f"\n✅ Berhasil! Data yang lebih presisi disimpan ke master_pelabuhan_8dpp_terfilter.csv")

Memulai ekstraksi pelabuhan spesifik untuk 8 DPP Coastal...
Mencari pelabuhan utama untuk Danau Toba (Area: Kabupaten Toba)...
Mencari pelabuhan utama untuk Kepulauan Seribu (Area: Kabupaten Administrasi Kepulauan Seribu)...
Mencari pelabuhan utama untuk Mandalika (Area: Kabupaten Lombok Barat)...
Mencari pelabuhan utama untuk Labuan Bajo (Area: Kabupaten Manggarai Barat)...
Mencari pelabuhan utama untuk Wakatobi (Area: Kabupaten Wakatobi)...
Gagal menarik data. Status Code: 429
Mencari pelabuhan utama untuk Morotai (Area: Kabupaten Pulau Morotai)...
Mencari pelabuhan utama untuk Tanjung Kelayang (Area: Kabupaten Belitung)...
Mencari pelabuhan utama untuk Likupang (Area: Kabupaten Minahasa Utara)...
Mencari pelabuhan utama untuk Raja Ampat (Area: Kabupaten Raja Ampat)...

=== HASIL EKSTRAKSI TITIK PELABUHAN (REVISI) ===
Empty DataFrame
Columns: [DPP, Wilayah_Kabupaten, Nama_Pelabuhan, Lat_Port, Lon_Port]
Index: []

✅ Berhasil! Data yang lebih presisi disimpan ke master_pelabuhan_8dpp_t

In [10]:
import pandas as pd

# 1. Load database pelabuhan hasil scraping-mu yang berisi 393 baris
df_raw_ports = pd.read_csv("master_data_pelabuhan_final.csv")

# 2. Ambil baris koordinat pelabuhan gerbang utama berdasarkan indeks spesifik yang valid
primary_ports_indices = [
    1,   # Tiga Raja Harbor (Danau Toba)
    62,  # Pelabuhan Muara Angke/Kali Adem (Kepulauan Seribu)
    91,  # Pelabuhan Ferry Lembar (Mandalika)
    295, # Pelabuhan Labuan Bajo (Labuan Bajo)
    155, # Pelabuhan Panggulubelo Wanci (Wakatobi)
    356, # Pelabuhan Ferry Daruba (Morotai)
    191, # Tanjung Pandan / Belitung Ferry (Tanjung Kelayang)
    259, # Likupang Munte Port (Likupang)
    261  # Pelabuhan Waisai (Raja Ampat)
]

# Ekstraksi baris terpilih
df_curated = df_raw_ports.iloc[primary_ports_indices].copy()

# 3. Berikan Label Nama DPP Resmi agar Sinkron dengan Data Hotel & Demand
dpp_labels = [
    "Danau Toba", "Kepulauan Seribu", "Mandalika", "Labuan Bajo", 
    "Wakatobi", "Morotai", "Tanjung Kelayang", "Likupang", "Raja Ampat"
]
df_curated['DPP'] = dpp_labels

# Perbaiki penamaan pelabuhan agar lebih profesional untuk dibaca di Dashboard
clean_names = [
    "Pelabuhan Tiga Raja", "Pelabuhan Muara Angke (Kali Adem)", "Pelabuhan Penyeberangan Lembar", 
    "Pelabuhan Utama Labuan Bajo", "Pelabuhan Panggulubelo Wanci", "Pelabuhan Penyeberangan Daruba",
    "Pelabuhan Tanjung Pandan", "Pelabuhan Munte Likupang", "Pelabuhan Utama Waisai"
]
df_curated['Nama_Pelabuhan'] = clean_names

# 4. Tambahkan Baris Khusus untuk Borobudur & Bromo (Landlocked / Tidak ada akses laut)
landlocked_data = pd.DataFrame([
    {"Wilayah": "Jawa Tengah", "Nama_Pelabuhan": "Tidak Ada Akses Laut", "Latitude": None, "Longitude": None, "DPP": "Borobudur"},
    {"Wilayah": "Jawa Timur", "Nama_Pelabuhan": "Tidak Ada Akses Laut", "Latitude": None, "Longitude": None, "DPP": "Bromo"}
])

df_curated_10dpp = pd.concat([df_curated, landlocked_data], ignore_index=True)

# Atur susunan kolom agar rapi
df_curated_10dpp = df_curated_10dpp[['DPP', 'Wilayah', 'Nama_Pelabuhan', 'Latitude', 'Longitude']]

# 5. SIMPAN SEBAGAI MASTER DATASET PORT TERFILTER
nama_output = "master_pelabuhan_10dpp_mengerucut.csv"
df_curated_10dpp.to_csv(nama_output, index=False)

print("=== MASTER DATA PELABUHAN 10 DPP (SANGAT MENGERUCUT) ===")
print(df_curated_10dpp.to_string())
print(f"\n✅ Berhasil menyaring data! File '{nama_output}' siap dipajang di Tableau.")

=== MASTER DATA PELABUHAN 10 DPP (SANGAT MENGERUCUT) ===
                 DPP                        Wilayah                     Nama_Pelabuhan  Latitude   Longitude
0         Danau Toba                 Sumatera Utara                Pelabuhan Tiga Raja  2.660992   98.930353
1   Kepulauan Seribu  Daerah Khusus Ibukota Jakarta  Pelabuhan Muara Angke (Kali Adem) -6.104333  106.771716
2          Mandalika            Nusa Tenggara Barat     Pelabuhan Penyeberangan Lembar -8.730002  116.077088
3        Labuan Bajo            Nusa Tenggara Timur        Pelabuhan Utama Labuan Bajo -8.493636   119.87536
4           Wakatobi              Sulawesi Tenggara       Pelabuhan Panggulubelo Wanci -5.338911   123.53354
5            Morotai                   Maluku Utara     Pelabuhan Penyeberangan Daruba  2.016692  128.280387
6   Tanjung Kelayang      Kepulauan Bangka Belitung           Pelabuhan Tanjung Pandan -2.746204  107.628483
7           Likupang                 Sulawesi Utara           Pelabuhan

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

print("Menghitung Bobot dan Skor Akhir Sisi Supply (Infrastruktur & Akses)...")

# 1. LOAD DATA MASTER SUPPLY YANG SUDAH BERSIH
# Gunakan file supply terakhir yang sudah kita hitung jarak pelabuhannya tadi
df_supply = pd.read_csv("master_supply_matrix_final_1.csv")
df_supply['Supply_Avg_Port_Dist_KM'] = df_supply['Supply_Avg_Port_Dist_KM'].fillna(0)

# 2. SELEKSI FITUR SUPPLY
fitur_supply = [
    'Supply_Total_Hotels', 'Supply_Avg_Hotel_Price', 'Supply_Avg_Hotel_Rating',
    'Supply_Avg_Airport_Dist_KM', 'Supply_Avg_Port_Dist_KM', 'Flight_Frequency'
]

# 3. NORMALISASI AWAL (0 sampai 1)
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_supply[fitur_supply]), columns=fitur_supply)

# 4. REKAYASA FITUR (INVERSI JARAK SPASIAL)
# Jarak dekat = Skor Bagus. Jarak jauh = Skor Jelek.
df_scaled['Supply_Avg_Airport_Dist_KM'] = 1 - df_scaled['Supply_Avg_Airport_Dist_KM']
df_scaled['Supply_Avg_Port_Dist_KM'] = 1 - df_scaled['Supply_Avg_Port_Dist_KM']

# 5. MENENTUKAN BOBOT SUPPLY (Expert Judgment Berbasis Konektivitas Parwis)
# Total bobot harus = 1.0 (100%)
bobot_supply = {
    'Supply_Total_Hotels': 0.20,       # Bobot 20% kapasitas kamar
    'Supply_Avg_Hotel_Price': 0.10,    # Bobot 10% keterjangkauan/level premium
    'Supply_Avg_Hotel_Rating': 0.10,   # Bobot 10% kualitas layanan hotel
    'Supply_Avg_Airport_Dist_KM': 0.20, # Bobot 20% kedekatan bandara udara
    'Supply_Avg_Port_Dist_KM': 0.15,    # Bobot 15% kedekatan pelabuhan laut
    'Flight_Frequency': 0.25           # Bobot 25% kepadatan jadwal penerbangan (paling krusial)
}

# 6. KALKULASI SKOR AKHIR SUPPLY (Skala 0 - 100)
skor_supply_kombinasi = np.zeros(len(df_supply))
for fitur, bobot in bobot_supply.items():
    skor_supply_kombinasi += df_scaled[fitur] * bobot

df_supply['Supply_Score'] = skor_supply_kombinasi * 100

print("\n=== HASIL SCORING SUPPLY MALAM INI ===")
print(df_supply[['DPP', 'Supply_Score']].sort_values(by='Supply_Score', ascending=False).to_string(index=False))

# Simpan hasil scoring malam ini
df_supply.to_csv("master_supply_scored.csv", index=False)
print("\n✅ Tugas Indra Selesai! File 'master_supply_scored.csv' berhasil dibuat.")

Menghitung Bobot dan Skor Akhir Sisi Supply (Infrastruktur & Akses)...

=== HASIL SCORING SUPPLY MALAM INI ===
             DPP  Supply_Score
       Mandalika     72.891400
       Borobudur     72.621483
     Labuan Bajo     68.778396
        Likupang     65.918305
      Raja Ampat     59.151650
      Danau Toba     54.585564
           Bromo     49.681423
        Wakatobi     41.970610
         Morotai     36.345732
Tanjung Kelayang     16.500616

✅ Tugas Indra Selesai! File 'master_supply_scored.csv' berhasil dibuat.


: 

In [6]:
import time
import re
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# ==========================================
# 1. DESTINASI (10 DPP)
# ==========================================
destinations = [
    "mandalika",
    "labuan bajo",
    "likupang",
    "danau toba",
    "borobudur",
    "bangka belitung",
    "wakatobi",
    "bromo",
    "raja ampat",
    "morotai"
]

# ==========================================
# 2. FUNCTION: EXTRACT LAT LONG DARI URL MAPS
# ==========================================
def extract_latlong_from_url(url):
    try:
        match = re.search(r'@(-?\d+\.\d+),(-?\d+\.\d+)', url)
        if match:
            return float(match.group(1)), float(match.group(2))
    except:
        pass
    return None, None

# ==========================================
# 3. CHROME SETUP
# ==========================================
options = Options()
options.add_argument("--start-maximized")
# Aktifkan baris di bawah jika tidak ingin jendela browser muncul terus-menerus
# options.add_argument("--headless") 

driver = webdriver.Chrome(options=options)
all_restaurant_data = []

# ==========================================
# 4. MAIN LOOP SCRAPER GOOGLE MAPS
# ==========================================
for destination in destinations:

    print("\n" + "=" * 60)
    print(f"SCRAPING LIVE GOOGLE MAPS: {destination.upper()}")
    print("=" * 60)

    # PERBAIKAN: Menggunakan URL Live Resmi Google Maps
    query = f"restoran di {destination}"
    url = f"https://www.google.com/maps/search/{query.replace(' ', '+')}"
    driver.get(url)
    
    time.sleep(6) # Waktu tunggu pemuatan peta pertama kali

    seen_restaurants = set()
    last_count = 0
    stuck_counter = 0
    MAX_STUCK = 4

    while True:
        try:
            # Mencari panel gulir tempat daftar restoran berada
            panel = driver.find_element(By.XPATH, '//div[@role="feed"]')
            driver.execute_script("arguments[0].scrollTo(0, arguments[0].scrollHeight);", panel)
        except Exception:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        
        time.sleep(2.5)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        restaurant_cards = soup.select("a.hfpxzc")
        current_count = len(restaurant_cards)
        
        print(f"Ditemukan sementara: {current_count} tempat makan.")

        if current_count == last_count:
            stuck_counter += 1
        else:
            stuck_counter = 0

        last_count = current_count

        if stuck_counter >= MAX_STUCK or "You've reached the end of the list" in driver.page_source:
            break

    # Ekstraksi Elemen Detail Restoran
    print(f"Mengekstrak data dari {len(restaurant_cards)} elemen HTML...")
    for card in restaurant_cards:
        try:
            restaurant_name = card.get("aria-label")
            restaurant_url = card.get("href")
            
            if not restaurant_name or (destination, restaurant_name) in seen_restaurants:
                continue
                
            seen_restaurants.add((destination, restaurant_name))
            parent_box = card.find_parent("div", class_="Nv2y1d")
            
            rating = None
            review_count = None
            price_level = None
            category = None

            if parent_box:
                # Ambil nilai rating bintang
                rating_tag = parent_box.select_one("span.MW4etd")
                if rating_tag:
                    rating = rating_tag.get_text(strip=True)

                # Ambil jumlah total review pembeli
                review_tag = parent_box.select_one("span.UY7F9")
                if review_tag:
                    review_raw = review_tag.get_text(strip=True)
                    review_count = review_raw.replace("(", "").replace(")", "").replace(",", "").replace(".", "")

                # Ambil kategori masakan dan tingkatan harga ($ / Rp)
                info_text_tag = parent_box.select_one("div.W4Efsd")
                if info_text_tag:
                    info_text = info_text_tag.get_text(" | ", strip=True)
                    price_match = re.search(r'(Rp\s?\d+|[$€]+)', info_text)
                    if price_match:
                        price_level = price_match.group(1)
                    
                    parts = [p.strip() for p in info_text.split("|") if p.strip()]
                    if len(parts) > 0:
                        category = parts[0]

            # Ambil koordinat dari teks URL peta
            lat, lon = None, None
            if restaurant_url:
                lat, lon = extract_latlong_from_url(restaurant_url)

            all_restaurant_data.append({
                "destination": destination,
                "restaurant_name": restaurant_name,
                "rating": rating,
                "review_count": review_count,
                "price_level": price_level,
                "category": category,
                "restaurant_url": restaurant_url,
                "latitude": lat,
                "longitude": lon
            })
        except:
            continue

driver.quit()

# ==========================================
# 5. INTEGRASI GEOPY FALLBACK (PELENGKAP DATA KOSONG)
# ==========================================
df_restaurant = pd.DataFrame(all_restaurant_data)
df_restaurant.drop_duplicates(subset=["destination", "restaurant_name"], inplace=True)

print("\n" + "-" * 50)
print("⚙️ MENJALANKAN GEOPY FALLBACK UNTUK KOORDINAT KOSONG")
print("-" * 50)

geolocator = Nominatim(user_agent="restaurant_locator_sdc")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Looping memeriksa baris data yang koordinatnya masih NaN
for idx, row in tqdm(df_restaurant.iterrows(), total=len(df_restaurant), desc="Mengisi Titik Spasial"):
    if pd.isna(row['latitude']) or pd.isna(row['longitude']):
        # Rencana A: Cari nama restoran + nama wilayah destinasi
        query_lengkap = f"{row['restaurant_name']} {row['destination']} Indonesia"
        try:
            location = geocode(query_lengkap)
            if location:
                df_restaurant.at[idx, 'latitude'] = location.latitude
                df_restaurant.at[idx, 'longitude'] = location.longitude
            else:
                # Rencana B: Jika spesifik gagal, pasang titik pusat kawasan destinasinya
                query_cadangan = f"{row['destination']} Indonesia"
                location_cadangan = geocode(query_cadangan)
                if location_cadangan:
                    df_restaurant.at[idx, 'latitude'] = location_cadangan.latitude
                    df_restaurant.at[idx, 'longitude'] = location_cadangan.longitude
        except:
            pass

# ==========================================
# 6. EXPORT KE MASTER PROJECT FOLDER
# ==========================================
output_path = r"C:\Users\User\Desktop\KULIAH\SEMESTER 6\SDC\kemenpar\google_restaurants_indonesia.csv"
df_restaurant.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"\n✅ Proses Selesai! Data restoran yang bersih dan lengkap berhasil disimpan di: {output_path}")


SCRAPING LIVE GOOGLE MAPS: MANDALIKA
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Mengekstrak data dari 8 elemen HTML...

SCRAPING LIVE GOOGLE MAPS: LABUAN BAJO
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Mengekstrak data dari 8 elemen HTML...

SCRAPING LIVE GOOGLE MAPS: LIKUPANG
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Mengekstrak data dari 8 elemen HTML...

SCRAPING LIVE GOOGLE MAPS: DANAU TOBA
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tempat makan.
Ditemukan sementara: 8 tem

Mengisi Titik Spasial: 100%|██████████| 80/80 [02:29<00:00,  1.86s/it]


✅ Proses Selesai! Data restoran yang bersih dan lengkap berhasil disimpan di: C:\Users\User\Desktop\KULIAH\SEMESTER 6\SDC\kemenpar\google_restaurants_indonesia.csv
